In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MagnitudeDropout(nn.Module):
    """
    Magnitude-aware dropout with per-sample normalization only.

    For x with shape [B, ...], we:
      - compute per-sample magnitudes |x|
      - convert them to probability mass q by normalizing across all non-batch dims
      - set keep probs: k = (1 - p) * [(1 - mix) + mix * M * q]
        where M = product of non-batch dimensions
      - sample mask ~ Bernoulli(k), and return y = x * mask / k   (unbiased)

    Args
    ----
    p   : float in [0,1), overall drop rate
    mix : float in [0,1], 0 => standard dropout; 1 => fully magnitude-based
    eps : small float for numerical stability
    """
    def __init__(self, p: float = 0.5, mix: float = 1.0, eps: float = 1e-8):
        super().__init__()
        if not (0.0 <= p < 1.0):
            raise ValueError("p must be in [0, 1)")
        if not (0.0 <= mix <= 1.0):
            raise ValueError("mix must be in [0, 1]")
        self.p = p
        self.mix = mix
        self.eps = eps

    def forward(self, x: torch.Tensor, p_override: float = None, mix_override: float = None) -> torch.Tensor:
        p = self.p if p_override is None else p_override
        mix = self.mix if mix_override is None else mix_override

        # no-op in eval or when p=0
        if (not self.training) or p == 0.0:
            return x

        if x.dim() == 1:
            # treat 1D as a single-sample batch for completeness
            x = x.unsqueeze(0)
            squeeze_back = True
        else:
            squeeze_back = False

        B = x.shape[0]
        # all non-batch dims
        non_batch_dims = tuple(range(1, x.dim()))

        # group size M (product of non-batch dims), broadcastable to [B, 1, 1, ...]
        M_val = 1
        for d in non_batch_dims:
            M_val *= x.shape[d]
        M = torch.as_tensor(M_val, device=x.device, dtype=x.dtype)

        mag = x.abs()
        # per-sample sum over non-batch dims, keepdim=True to broadcast back
        S = mag.sum(dim=non_batch_dims, keepdim=True)  # shape [B, 1, 1, ...]
        uniform_q = (1.0 / M).to(dtype=x.dtype, device=x.device)

        # q: per-sample probability mass over non-batch dims
        q = torch.where(S > self.eps, mag / (S + self.eps), uniform_q)

        # keep probabilities
        k = (1.0 - p) * ((1.0 - mix) + mix * M * q)
        k = torch.clamp(k, min=0.0, max=1.0 - 1e-6)

        mask = torch.bernoulli(k)
        y = x * mask / (k + self.eps)

        if squeeze_back:
            y = y.squeeze(0)
        return y


In [2]:
import torch.nn as nn
import torch.nn.functional as F

class DropoutModel(nn.Module):
    def __init__(self):
        super(DropoutModel, self).__init__()
        self.fc = nn.ModuleList([
            nn.Linear(784, 500),
            nn.Linear(500, 50),
            nn.Linear(50, 10)
        ])

    def forward(self, input, p=0):
        result = input
        for i, layer in enumerate(self.fc):
            result = F.elu(layer(result))
            if i < len(self.fc) - 1:
                result = F.dropout(result, p, training=True)
        return result

    def loss(self, **kwargs):
        out = self(kwargs['input'], kwargs['p'])
        return F.cross_entropy(out, kwargs['target'], size_average=kwargs['average'])

class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(784, 500),
            nn.ELU(),
            nn.Linear(500, 50),
            nn.ELU(),
            nn.Linear(50, 10)
        )

    def forward(self, input):
        return self.fc(input)

    def loss(self, **kwargs):
        out = self(kwargs['input'])
        return F.cross_entropy(out, kwargs['target'], size_average=kwargs['average'])


In [3]:
class MagDropoutModel(nn.Module):
    def __init__(self, mix: float = 1.0):
        super().__init__()
        self.fc1 = nn.Linear(784, 500)
        self.fc2 = nn.Linear(500, 50)
        self.fc3 = nn.Linear(50, 10)

        self.magdrop = MagnitudeDropout(p=0.5, mix=mix)

    def forward(self, x, p=0.0, mix=None):
        h1 = F.elu(self.fc1(x))
        h1 = self.magdrop(h1, p_override=p, mix_override=(self.magdrop.mix if mix is None else mix))

        h2 = F.elu(self.fc2(h1))
        h2 = self.magdrop(h2, p_override=p, mix_override=(self.magdrop.mix if mix is None else mix))

        out = self.fc3(h2)
        return out

    def loss(self, **kwargs):
        out = self(kwargs['input'], kwargs.get('p', 0.0), kwargs.get('mix', None))
        return F.cross_entropy(out, kwargs['target'], size_average=kwargs['average'])


In [4]:
import time
import math
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import datasets

DEVICE = 'cuda' if t.cuda.is_available() else 'cpu'

def make_zero_fraction_recorder(layer_names):
    """
    Returns: dict for storage + a hook factory that measures (tensor == 0).float().mean().
    Use it on the post-dropout tensors.
    """
    store = {name: [] for name in layer_names}

    def hook_for(name):
        def _hook(module, inp, out):
            with t.no_grad():
                frac_zero = (out == 0).float().mean().item()
                store[name].append(frac_zero)
        return _hook

    return store, hook_for

def train_eval_model(model,
                     train_loader,
                     test_loader,
                     num_epochs=5,
                     lr=5e-4,
                     mode='simple',
                     p_train=0.4,
                     p_val=0.0,
                     mix=1.0):
    """
    mode in {'simple', 'dropout', 'magdropout'} controls forward signature and zero-fraction registration.
    Returns logs: {'train_loss': [...], 'val_loss': [...], 'zeros': {'layer1':[...], 'layer2':[...], ...}}
    """
    model = model.to(DEVICE)
    opt = t.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(reduction='mean')

    # Register hooks to measure zeros after *dropout* (or after ELU for simple)
    layer_names = []
    hooks = []
    zeros_store, hook_for = make_zero_fraction_recorder(['h1', 'h2'])

    # For DropoutModel/MagDropoutModel we'll attach hooks by wrapping forward pass
    # (because dropout is applied inline). For SimpleModel we’ll hook after each ELU.

    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (x, y) in enumerate(train_loader):
            x = x.view(-1, 784).to(DEVICE)
            y = y.to(DEVICE)

            opt.zero_grad()

            # Attach temporary hooks for this pass to capture zeros *after* dropout
            # We do it by grabbing intermediate tensors manually
            if mode == 'simple':
                # forward that exposes the two hidden activations
                h1 = F.elu(model.fc[0](x))
                h2 = F.elu(model.fc[2](h1))
                # record zeros (expected ~0)
                zeros_store['h1'].append((h1 == 0).float().mean().item())
                zeros_store['h2'].append((h2 == 0).float().mean().item())
                logits = model.fc[4](h2)
            elif mode == 'dropout':
                # replicate DropoutModel forward with access points
                h1 = F.elu(model.fc[0](x))
                h1 = F.dropout(h1, p_train, training=True)
                zeros_store['h1'].append((h1 == 0).float().mean().item())
                h2 = F.elu(model.fc[1](h1))
                h2 = F.dropout(h2, p_train, training=True)
                zeros_store['h2'].append((h2 == 0).float().mean().item())
                logits = model.fc[2](h2)
            elif mode == 'magdropout':
                h1 = F.elu(model.fc[0](x))
                h1 = model.magdrop(h1, p_override=p_train, mix_override=mix)
                zeros_store['h1'].append((h1 == 0).float().mean().item())
                h2 = F.elu(model.fc[1](h1))
                h2 = model.magdrop(h2, p_override=p_train, mix_override=mix)
                zeros_store['h2'].append((h2 == 0).float().mean().item())
                logits = model.fc[2](h2)
            else:
                raise ValueError("mode must be 'simple' | 'dropout' | 'magdropout'")

            loss = criterion(logits, y)
            loss.backward()
            opt.step()

            train_losses.append(loss.detach().item())

        # Validation
        model.eval()
        with t.no_grad():
            total_loss, total_count = 0.0, 0
            for x, y in test_loader:
                x = x.view(-1, 784).to(DEVICE)
                y = y.to(DEVICE)

                if mode == 'simple':
                    out = model(x)
                elif mode == 'dropout':
                    # use p_val = 0.0 to disable dropout at validation
                    h1 = F.elu(model.fc[0](x))
                    h1 = F.dropout(h1, p_val, training=True)
                    h2 = F.elu(model.fc[1](h1))
                    h2 = F.dropout(h2, p_val, training=True)
                    out = model.fc[2](h2)
                else:  # magdropout
                    h1 = F.elu(model.fc[0](x))
                    h1 = model.magdrop(h1, p_override=p_val, mix_override=mix)
                    h2 = F.elu(model.fc[1](h1))
                    h2 = model.magdrop(h2, p_override=p_val, mix_override=mix)
                    out = model.fc[2](h2)

                total_loss += criterion(out, y).item() * x.size(0)
                total_count += x.size(0)

            val_losses.append(total_loss / total_count)

    logs = {
        'train_loss': train_losses,
        'val_loss': val_losses,
        'zeros': zeros_store
    }
    return logs


In [5]:
from torchvision import datasets, transforms

def get_mnist_loaders(batch_size=1000, data_root='./data'):
    tfm = transforms.ToTensor()
    train_ds = datasets.MNIST(root=data_root, train=True, download=True, transform=tfm)
    test_ds  = datasets.MNIST(root=data_root, train=False, download=True, transform=tfm)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, test_loader

train_loader, test_loader = get_mnist_loaders(batch_size=1000, data_root='./data')


RuntimeError: Error downloading train-images-idx3-ubyte.gz:
Tried http://yann.lecun.com/exdb/mnist/, got:
HTTP Error 404: Not Found
Tried https://ossci-datasets.s3.amazonaws.com/mnist/, got:
<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)>


In [6]:
EPOCHS = 10
LR = 5e-4
P_TRAIN = 0.4
P_VAL = 0.0
MIX = 1.0   # set 0.0 to exactly recover standard dropout as a sanity check

# Simple
simple_model = SimpleModel()
logs_simple = train_eval_model(simple_model, train_loader, test_loader,
                               num_epochs=EPOCHS, lr=LR, mode='simple')

# Standard dropout
std_model = DropoutModel()
logs_std = train_eval_model(std_model, train_loader, test_loader,
                            num_epochs=EPOCHS, lr=LR, mode='dropout', p_train=P_TRAIN, p_val=P_VAL)

# Magnitude-aware dropout
mag_model = MagDropoutModel(mix=MIX)
logs_mag = train_eval_model(mag_model, train_loader, test_loader,
                            num_epochs=EPOCHS, lr=LR, mode='magdropout', p_train=P_TRAIN, p_val=P_VAL, mix=MIX)


NameError: name 'train_loader' is not defined

In [7]:
import matplotlib.pyplot as plt
import numpy as np

# Training loss (iteration-wise)
plt.figure()
plt.plot(np.arange(len(logs_simple['train_loss'])), logs_simple['train_loss'], label='simple')
plt.plot(np.arange(len(logs_std['train_loss'])), logs_std['train_loss'], label='dropout (p=0.4)')
plt.plot(np.arange(len(logs_mag['train_loss'])), logs_mag['train_loss'], label='magdropout (p=0.4)')
plt.xlabel('Training iteration')
plt.ylabel('Cross-entropy loss')
plt.title('Training loss (per iteration)')
plt.legend()
plt.show()

# Validation loss (per epoch)
plt.figure()
plt.plot(np.arange(1, len(logs_simple['val_loss'])+1), logs_simple['val_loss'], label='simple')
plt.plot(np.arange(1, len(logs_std['val_loss'])+1), logs_std['val_loss'], label='dropout (p=0.4)')
plt.plot(np.arange(1, len(logs_mag['val_loss'])+1), logs_mag['val_loss'], label='magdropout (p=0.4)')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Validation loss (per epoch)')
plt.legend()
plt.show()


NameError: name 'logs_simple' is not defined

<Figure size 640x480 with 0 Axes>

In [8]:
def plot_zero_fractions(logs, title_prefix):
    # logs['zeros'] has keys 'h1', 'h2'
    plt.figure()
    plt.plot(np.arange(len(logs['zeros']['h1'])), logs['zeros']['h1'], label='layer h1')
    plt.plot(np.arange(len(logs['zeros']['h2'])), logs['zeros']['h2'], label='layer h2')
    plt.xlabel('Training iteration')
    plt.ylabel('Fraction zeroed')
    plt.title(f'{title_prefix}: fraction of activations zeroed (per iteration)')
    plt.legend()
    plt.show()

# Simple (usually ~0 for ELU)
plot_zero_fractions(logs_simple, 'Simple')

# Standard dropout
plot_zero_fractions(logs_std, 'Standard dropout (p=0.4)')

# Magnitude-aware dropout
plot_zero_fractions(logs_mag, 'Magnitude-aware dropout (p=0.4)')


NameError: name 'logs_simple' is not defined

In [9]:
def aggregate_by_epoch(iter_series, num_iters_per_epoch):
    arr = np.array(iter_series, dtype=float)
    n = len(arr)
    epochs = n // num_iters_per_epoch
    arr = arr[:epochs * num_iters_per_epoch].reshape(epochs, num_iters_per_epoch)
    return arr.mean(axis=1)

# estimate iters/epoch from a run (using standard run as reference)
iters_per_epoch = len(logs_std['zeros']['h1']) // EPOCHS

h1_std_epoch = aggregate_by_epoch(logs_std['zeros']['h1'], iters_per_epoch)
h2_std_epoch = aggregate_by_epoch(logs_std['zeros']['h2'], iters_per_epoch)

h1_mag_epoch = aggregate_by_epoch(logs_mag['zeros']['h1'], iters_per_epoch)
h2_mag_epoch = aggregate_by_epoch(logs_mag['zeros']['h2'], iters_per_epoch)

plt.figure()
plt.plot(np.arange(1, len(h1_std_epoch)+1), h1_std_epoch, label='std h1')
plt.plot(np.arange(1, len(h2_std_epoch)+1), h2_std_epoch, label='std h2')
plt.plot(np.arange(1, len(h1_mag_epoch)+1), h1_mag_epoch, label='mag h1')
plt.plot(np.arange(1, len(h2_mag_epoch)+1), h2_mag_epoch, label='mag h2')
plt.xlabel('Epoch')
plt.ylabel('Mean fraction zeroed')
plt.title('Mean fraction zeroed per epoch (std vs mag)')
plt.legend()
plt.show()


NameError: name 'logs_std' is not defined